In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 12. Tokenizer — text text ⭐

> **Learning Goals**
> - text text text LLMtext text text
> - BPE text text Pythontext text
> - WordPiece, Unigram Language Modeltext text text
> - text text Tokenizer(Byte-level BPE)text text text

## 12.1 text text

text text text:
- **text text**: "Hello, world!" → ["Hello", ",", "world", "!"]
  - Problem: text text, text/text text text, OOV(Out-Of-Vocabulary)
- **text text**: "Hello" → ['H', 'e', 'l', 'l', 'o']
  - Problem: text text text, text text text
- **text text**: "unhappiness" → ["un", "happiness"] text ["un", "happ", "iness"]
  - **LLM text**: textdegrees text text text, text text text

GPT-2: Byte-level BPE. BERT: WordPiece. T5: Unigram.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

# text text text text
text = "The quick brown fox jumps over the lazy dog."
word_tokens = text.lower().replace('.', ' .').split()
print(f"word text: {word_tokens}")

# Character-level
char_tokens = list(text.lower())
print(f"Character-level: {char_tokens}")

# text (Concepttext text)
subword_tokens = ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']
print(f"text text (text): {subword_tokens}")


## 12.2 Byte-Pair Encoding (BPE) — GPT-2text text

**text**:
1. text text text text text
2. text textdegreestext text text text(2-gram)text text
3. text text degreestext text text

text: 'low' (5text), 'lower' (2text), 'newest' (6text), 'widest' (3text)

textdegrees text 'e'+'s' text 9text → text → 'es'


In [ ]:
# BPE text text
from collections import Counter

class BPE:
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.merges = []  # (a, b) text text text
        self.vocab = set()

    def _get_word_freqs(self, corpus):
        """word textdegrees Computation."""
        word_freqs = Counter()
        for text in corpus:
            words = text.lower().split()
            for w in words:
                word_freqs[w] += 1
        return word_freqs

    def _get_pair_freqs(self, splits, word_freqs):
        """text text textdegrees Computation."""
        pair_freqs = Counter()
        for word, split in splits.items():
            for i in range(len(split) - 1):
                pair = (split[i], split[i+1])
                pair_freqs[pair] += word_freqs[word]
        return pair_freqs

    def _merge(self, pair, splits):
        """text wordtext pairtext textSum."""
        new_splits = {}
        a, b = pair
        for word, split in splits.items():
            new_split = []
            i = 0
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            new_splits[word] = new_split
        return new_splits

    def train(self, corpus):
        word_freqs = self._get_word_freqs(corpus)
        # text wordtext Character-leveltext Decomposition (text </w> Display)
        splits = {w: list(w) + ['</w>'] for w in word_freqs}
        # text text
        self.vocab = set(c for w in word_freqs for c in w) | {'</w>'}

        for _ in range(self.num_merges):
            pair_freqs = self._get_pair_freqs(splits, word_freqs)
            if not pair_freqs:
                break
            best_pair = max(pair_freqs, key=pair_freqs.get)
            splits = self._merge(best_pair, splits)
            self.merges.append(best_pair)
            self.vocab.add(best_pair[0] + best_pair[1])

    def encode(self, word):
        """Trainingtext text word Encoding."""
        split = list(word) + ['</w>']
        for a, b in self.merges:
            i = 0
            new_split = []
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            split = new_split
        return split

# text text Training
corpus = [
    "low low low low low",
    "lower lower newest newest newest",
    "widest widest newest newest",
]
bpe = BPE(num_merges=10)
bpe.train(corpus)

print("Trainingtext textSum text:")
for i, (a, b) in enumerate(bpe.merges, 1):
    print(f"  {i}. {a} + {b} -> {a+b}")
print(f"\nVocabulary Size: {len(bpe.vocab)}")

# Encoding Test
for word in ['low', 'lowest', 'newer', 'widest']:
    tokens = bpe.encode(word)
    print(f"  {word:>10} -> {tokens}")


## 12.3 WordPiece — BERTtext text

BPEtext text **text text text**:

$$\text{score}(a, b) = \frac{\text{freq}(ab)}{\text{freq}(a) \cdot \text{freq}(b)}$$

BPEtext textdegrees text text, WordPiecetext textdegreestext text text text.
text text text text text text text, text text text text text.


In [ ]:
# WordPiece text text
class WordPiece:
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.merges = []

    def _get_pair_scores(self, splits, word_freqs):
        pair_freqs = Counter()
        char_freqs = Counter()
        for word, split in splits.items():
            for i in range(len(split) - 1):
                pair = (split[i], split[i+1])
                pair_freqs[pair] += word_freqs[word]
            for s in split:
                char_freqs[s] += word_freqs[word]
        scores = {p: f / (char_freqs[p[0]] * char_freqs[p[1]]) for p, f in pair_freqs.items()}
        return scores

    def _merge(self, pair, splits):
        new_splits = {}
        a, b = pair
        for word, split in splits.items():
            new_split = []
            i = 0
            while i < len(split):
                if i < len(split) - 1 and split[i] == a and split[i+1] == b:
                    new_split.append(a + b)
                    i += 2
                else:
                    new_split.append(split[i])
                    i += 1
            new_splits[word] = new_split
        return new_splits

    def train(self, corpus):
        word_freqs = Counter()
        for text in corpus:
            for w in text.lower().split():
                word_freqs[w] += 1
        # WordPiecetext ## text text text text text
        splits = {}
        for w in word_freqs:
            chars = list(w)
            splits[w] = [chars[0]] + ['##' + c for c in chars[1:]]
        for _ in range(self.num_merges):
            scores = self._get_pair_scores(splits, word_freqs)
            if not scores:
                break
            best = max(scores, key=scores.get)
            splits = self._merge(best, splits)
            self.merges.append(best)

wp = WordPiece(num_merges=10)
wp.train(corpus)
print("WordPiece Trainingtext textSum:")
for i, (a, b) in enumerate(wp.merges, 1):
    print(f"  {i}. {a} + {b} -> {a+b}")


## 12.4 Unigram Language Model — T5text text

BPE/WordPiecetext **bottom-up** (text text). Unigramtext **top-down** (text text text).

1. text text = text text text (text)
2. text text text $P(t) = \text{count}(t) / \sum \text{count}$
3. text $\mathcal{L} = -\sum_i \log P(\mathbf{x}_i) = -\sum_i \log \sum_{\text{seg}} \prod P(t)$
4. text text text text text text
5. text

EM text text text text.


In [ ]:
# Unigram text text (text)
from itertools import combinations

class UnigramTokenizer:
    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        self.vocab = {}  # token -> prob

    def _get_candidates(self, word_freqs, max_len=10):
        candidates = Counter()
        for word, freq in word_freqs.items():
            for i in range(len(word)):
                for j in range(i+1, min(i+max_len+1, len(word)+1)):
                    candidates[word[i:j]] += freq
        return candidates

    def _segment(self, word, vocab):
        """Viterbitext Optimal text."""
        n = len(word)
        # dp[i] = (best_log_prob, best_segmentation)
        dp = [(-float('inf'), None)] * (n + 1)
        dp[0] = (0, [])
        for i in range(1, n + 1):
            for j in range(max(0, i - 10), i):
                token = word[j:i]
                if token in vocab:
                    log_p = np.log(vocab[token])
                    if dp[j][0] + log_p > dp[i][0]:
                        dp[i] = (dp[j][0] + log_p, dp[j][1] + [token])
        return dp[n]

    def train(self, corpus):
        word_freqs = Counter()
        for text in corpus:
            for w in text.lower().split():
                word_freqs[w] += 1
        candidates = self._get_candidates(word_freqs)
        total = sum(candidates.values())
        self.vocab = {t: c / total for t, c in candidates.items()}

        # Iterationtext Loss Increasetext text text text
        while len(self.vocab) > self.vocab_size:
            # text text text text Loss Increase Computation
            losses = {}
            for token in list(self.vocab.keys()):
                if len(token) == 1:  # text text text
                    continue
                # text text text Loss (text)
                losses[token] = self.vocab[token]  # text Probabilitytext losstext
            if not losses:
                break
            # Losstext text text text text
            sorted_tokens = sorted(losses.items(), key=lambda x: x[1])
            n_remove = max(1, len(self.vocab) - self.vocab_size)
            for token, _ in sorted_tokens[:n_remove]:
                del self.vocab[token]

# text text
uni = UnigramTokenizer(vocab_size=30)
uni.train(corpus)
print(f"Unigram Vocabulary Size: {len(uni.vocab)}")
print(f"text text: {sorted(uni.vocab.items(), key=lambda x: -x[1])[:10]}")

# text text
for word in ['lowest', 'newest']:
    log_p, seg = uni._segment(word, uni.vocab)
    print(f"  {word} -> {seg} (log_p={log_p:.4f})")


## 12.5 Byte-level BPE — GPT-2/3text text

Problem: text text text text (text 11,172text, text text).
text: text text **UTF-8 text**text Encoding text BPE text.

- text 256text → text text 256
- UNK(unknown) text text — text text text text
- text text text

GPT-2 text: 256 text + 50,257 text ≈ 50K text


In [ ]:
# Byte-level BPE text
text = "Hello, text!"
bytes_seq = text.encode('utf-8')
print(f"Original: {text}")
print(f"UTF-8 text: {list(bytes_seq)}")
print(f"text text: {len(bytes_seq)} (text text {len(text)})")
print("\n=> text text text text text. UNK text.")

# HuggingFace tokenizers text text
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace
except ImportError:
    print("\n[HuggingFace tokenizers text text: pip install tokenizers]")

# text BPE Tokenizer (HuggingFace)
try:
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=200, special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"])

    # text text Training
    from llm_math.data import load_tiny_shakespeare
    train_text = load_tiny_shakespeare(max_chars=5000)
    tokenizer.train_from_iterator([train_text], trainer)

    print(f"\nTrainingtext Vocabulary Size: {tokenizer.get_vocab_size()}")
    # Encoding Test
    enc = tokenizer.encode("To be or not to be")
    print(f"'To be or not to be' -> {enc.tokens}")
    print(f"IDs: {enc.ids}")
except ImportError:
    print("\n[tokenizers text text. pip install tokenizerstext text]")


## 12.6 [CPU/GPU Benchmark ④] Tokenizer Training/Encoding Speed

Tokenizer text CPUtext text, text text text text Speed text text.


In [ ]:
# Tokenizer Benchmark (CPUtext)
import time
from llm_math.data import load_tiny_shakespeare

# text text text
text = load_tiny_shakespeare(max_chars=100000)
print(f"Benchmarktext text: {len(text):,}text")

# 1. text BPEtext Training Time text
def bench_bpe_train(text, n_merges, repeat=3):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        bpe = BPE(num_merges=n_merges)
        bpe.train([text])
        times.append(time.perf_counter() - t0)
    return np.mean(times), np.std(times)

print(f"\n{'merges':>8} | {'train (s)':>12}")
print("-" * 25)
for n in [10, 50, 100, 200]:
    m, s = bench_bpe_train(text, n, repeat=2)
    print(f"{n:>8} | {m:>10.4f}±{s:.4f}")

# 2. Encoding Speed
bpe_small = BPE(num_merges=50)
bpe_small.train([text[:5000]])

def bench_encode(bpe, text, n_words=1000):
    words = text.split()[:n_words]
    t0 = time.perf_counter()
    for w in words:
        bpe.encode(w)
    return (time.perf_counter() - t0) * 1000

t_ms = bench_encode(bpe_small, text, n_words=1000)
print(f"\n1000text Encoding: {t_ms:.2f}ms ({t_ms/1000:.3f}ms/text)")
print("\n=> Tokenizertext CPU text. text textPlane text text.")
print("   HuggingFace tokenizerstext Rust Implementationtext text text.")


## 12.7 text text Performance text

- text text → text text, Model text, text text text
- text text → text text, text text text, text text Training text

LLMtext text text:
- GPT-2: 50,257
- GPT-3: 50,257
- LLaMA-2: 32,000
- LLaMA-3: 128,256
- GPT-4: ~100,000


In [ ]:
# text text Sequence Length Comparison (text)
# text text text text text text
np.random.seed(0)
text_sample = load_tiny_shakespeare(max_chars=5000)

# text text text Trainingtext text text text Comparison
vocab_sizes = [30, 50, 100, 200, 500]
avg_tokens = []
for vs in vocab_sizes:
    bpe = BPE(num_merges=vs - 30)  # text text text 30
    bpe.train([text_sample])
    # text text text text text
    sample_words = text_sample.split()[:100]
    total_tokens = sum(len(bpe.encode(w)) for w in sample_words)
    avg_tokens.append(total_tokens / len(sample_words))

plt.figure(figsize=(9, 5))
plt.plot(vocab_sizes, avg_tokens, 'o-', linewidth=2, markersize=8)
plt.xlabel('Vocabulary Size')
plt.ylabel('Mean text text / word')
plt.title('Vocabulary Sizetext Sequence Lengthtext Trade-off')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch12_vocab_size_tradeoff.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> text text wordtext text text text. text Embedding text text.")


## 12.8 Key Takeaways

| text | text text | text Model |
|---|---|---|
| BPE | textdegrees text | GPT-2 |
| WordPiece | $\frac{\text{freq}(ab)}{\text{freq}(a)\text{freq}(b)}$ | BERT |
| Unigram | text text (top-down) | T5 |
| Byte-level BPE | BPE on bytes | GPT-2/3/4 |

## Exercises

1. text text BPEtext text text text, text text text.
2. BPEtext WordPiecetext text text Trainingtext text text Comparisontext.
3. text text 100, 500, 1000text text text text text Comparisontext.
4. Byte-level BPEtext text UNK text text text.
5. HuggingFace `tokenizers` text GPT-2 Tokenizertext text text text Encodingtext.

> Solutions: `solutions/ch12_solutions.ipynb`
